# Lesson 24 Lab — Benchmark Design: Throughput, Latency, Concurrency, and Memory

**Puzzle:** How can the same GPU path improve throughput while worsening latency?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Benchmark outputs include latency distribution, throughput, concurrency, queueing, TTFT, token latency, memory, power/cost, and workload shape. They cannot be collapsed into one number.

### Core mechanism

For a fixed operator, throughput is `batch / latency`; batching can raise throughput while each item waits longer. In a service, arrival rate and queueing add latency beyond GPU execution.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "24-benchmark-design"
device = require_cuda()
torch.manual_seed(2026 + 24)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

A configuration optimized for batch throughput may violate interactive p99. More concurrency improves utilization until memory pressure or scheduling raises tails.

### What this code tests

The lab sweeps batch size and reports median, p90, examples/s, and peak allocated memory; it labels the result as an operator workload, not a server test.

**Experiment:** Benchmark a CUDA MLP over several batch sizes, recording median, p90, examples per second, and peak allocated memory.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
model=torch.nn.Sequential(torch.nn.Linear(2048,4096,bias=False),torch.nn.GELU(),torch.nn.Linear(4096,2048,bias=False)).to(device).bfloat16(); rows=[]
for batch in (1,8,32,128):
    x=torch.randn(batch,2048,device=device,dtype=torch.bfloat16); torch.cuda.reset_peak_memory_stats(); timing=cuda_benchmark(lambda:model(x),warmup=5,repeats=20)
    rows.append({"batch":batch,"timing":timing,"examples_per_second":round(batch/(timing["median_ms"]/1000),2),
                 "peak_allocated_mib":round(torch.cuda.max_memory_allocated()/2**20,3)})
result=base_result(24,"pytorch-gpu"); result.update({"operator_workload":rows,
    "conclusion":"Batching changed throughput, latency, and memory in different directions; no service queueing was modeled."})


## 3. Inspect the evidence

Compare all axes at the same shape and precision. This is an operator workload, not a vLLM service benchmark.

### Acceptance and rollback gate

Declare workload distribution, warm-up, repetitions, synchronization, concurrency, percentile method, precision, and SLO before seeing the candidate.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Batching changed throughput, latency, and memory in different directions; no service queueing was modeled.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:12+00:00",
  "lesson": 24,
  "operator_workload": [
    {
      "batch": 1,
      "examples_per_second": 21656.27,
      "peak_allocated_mib": 64.023,
      "timing": {
        "median_ms": 0.046176,
        "p90_ms": 0.048736,
        "repeats": 20,
        "samples_ms": [
          0.0648,
          0.054496,
          0.048736,
          0.046016,
          0.04672,
          0.046752,
          0.04592,
          0.046144,
          0.04768,
          0.046496,
          0.045792,
          0.046688,
          0.046816,
          0.045568,
          0.045632,
          0.04544,
        

## 4. Explain the result

Choose a candidate against a service-level objective, not the single largest throughput number.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).